# Benutzerdefiniertes CNN ohne keras.models oder keras.layers

Dieses Notebook implementiert ein Convolutional Neural Network (CNN) ohne Verwendung von keras.models oder keras.layers zur Erkennung von Autos im CIFAR-10 Datensatz.

## Einführung und theoretischer Hintergrund

Convolutional Neural Networks (CNNs) haben sich als leistungsstarke Werkzeuge für die Bildverarbeitung und -erkennung etabliert. Während Frameworks wie Keras und TensorFlow die Implementierung von CNNs erheblich vereinfachen, ist es für ein tieferes Verständnis wertvoll, die grundlegenden Operationen und Algorithmen selbst zu implementieren.

In diesem Notebook implementieren wir ein CNN von Grund auf mit NumPy, ohne die höheren Abstraktionen von keras.models oder keras.layers zu verwenden. Dies ermöglicht uns:

1. Ein tieferes Verständnis der mathematischen Grundlagen von CNNs zu entwickeln
2. Die Vorwärts- und Rückwärtspropagierung für jede Schicht explizit zu implementieren
3. Den Lernprozess und die Parameteraktualisierung im Detail zu verstehen
4. Die Herausforderungen und Komplexitäten bei der Implementierung von neuronalen Netzwerken zu erfahren

Die Hauptkomponenten unserer Implementierung umfassen:

- **Faltungsoperation (Convolution)**: Die grundlegende Operation in CNNs, bei der Filter über das Eingabebild gleiten, um Merkmale zu extrahieren. Mathematisch ist dies eine Kreuzkorrelation, bei der jeder Ausgabepixel durch die gewichtete Summe der Eingabepixel in einem lokalen Bereich berechnet wird.

- **Aktivierungsfunktionen**: Nichtlineare Funktionen wie ReLU (Rectified Linear Unit), die es dem Netzwerk ermöglichen, komplexe Muster zu lernen. Die ReLU-Funktion ist definiert als $f(x) = \max(0, x)$.

- **Pooling**: Eine Operation zur Reduzierung der räumlichen Dimensionen, typischerweise durch Auswahl des maximalen Wertes in einem definierten Fenster (Max-Pooling).

- **Vollständig verbundene Schichten (Fully Connected)**: Schichten, in denen jedes Neuron mit allen Neuronen der vorherigen Schicht verbunden ist, ähnlich wie in traditionellen neuronalen Netzwerken.

- **Backpropagation**: Der Algorithmus zur Berechnung der Gradienten der Verlustfunktion in Bezug auf die Netzwerkparameter, der es ermöglicht, die Parameter durch Gradientenabstieg zu aktualisieren.

Diese Implementierung ist zwar weniger effizient als optimierte Bibliotheken wie TensorFlow, bietet aber wertvolle Einblicke in die innere Funktionsweise von CNNs.

## Überblick über die Schritte
- Implementierung eines CNN von Grund auf mit NumPy
- Implementierung der Vorwärts- und Rückwärtspropagierung für alle Schichten
- Training des Modells mit Mini-Batch Gradient Descent
- Evaluierung des Modells auf Testdaten
- Visualisierung der Ergebnisse

## Importieren der benötigten Bibliotheken

Für unsere benutzerdefinierte CNN-Implementierung benötigen wir nur wenige Bibliotheken:

- **numpy**: Für effiziente numerische Operationen und Array-Manipulationen
- **matplotlib**: Für die Visualisierung der Trainingsergebnisse und Vorhersagen
- **os**: Für Dateisystem-Operationen wie das Laden der vorbereiteten Daten
- **time**: Für die Messung der Trainingszeit

Im Gegensatz zur Keras-Implementierung verwenden wir keine spezialisierten Deep-Learning-Bibliotheken, da wir alle Funktionalitäten selbst implementieren werden.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import time

## Vorbereitung der Verzeichnisse und Laden der Daten

Bevor wir mit der Implementierung beginnen, laden wir die vorbereiteten Daten aus dem ersten Notebook. Diese Daten wurden bereits normalisiert und mit binären Labels versehen (1 für Auto, 0 für Nicht-Auto).

Da unsere benutzerdefinierte Implementierung rechenintensiver ist als die Keras-Version, reduzieren wir die Datensatzgröße für das Training, um die Ausführungszeit zu verkürzen. Dies ist ein üblicher Ansatz beim Prototyping und Testen von Algorithmen.

In [ ]:
# Vorbereitung der Verzeichnisse
data_dir = '../data'
models_dir = '../models'
custom_models_dir = os.path.join(models_dir, 'custom_cnn')
visualizations_dir = os.path.join(models_dir, 'visualizations')

os.makedirs(custom_models_dir, exist_ok=True)
os.makedirs(visualizations_dir, exist_ok=True)

# Laden der vorbereiteten Daten
x_train = np.load(os.path.join(data_dir, 'x_train_normalized.npy'))
x_test = np.load(os.path.join(data_dir, 'x_test_normalized.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train_binary.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test_binary.npy'))

print(f"Trainingsbilder: {x_train.shape}")
print(f"Testbilder: {x_test.shape}")
print(f"Trainings-Labels: {y_train.shape}")
print(f"Test-Labels: {y_test.shape}")

## Reduzieren der Datensatzgröße für schnelleres Training

Da unsere benutzerdefinierte CNN-Implementierung nicht die optimierten Operationen von Frameworks wie TensorFlow nutzt, wäre das Training auf dem gesamten Datensatz sehr zeitaufwändig. Daher reduzieren wir die Datensatzgröße für das Training.

Wir wählen eine Teilmenge der Trainings- und Testdaten aus, wobei wir darauf achten, dass die Klassenverteilung (Autos vs. Nicht-Autos) erhalten bleibt. Dies ermöglicht uns, die Implementierung zu testen und zu validieren, ohne übermäßig lange Rechenzeiten in Kauf nehmen zu müssen.

In einer Produktionsumgebung würde man natürlich den vollständigen Datensatz verwenden und optimierte Implementierungen nutzen.

In [ ]:
# Reduzieren der Datensatzgröße für schnelleres Training
def reduce_dataset(x_data, y_data, num_samples, random_seed=42):
    np.random.seed(random_seed)
    
    # Indizes für jede Klasse finden
    car_indices = np.where(y_data == 1)[0]
    non_car_indices = np.where(y_data == 0)[0]
    
    # Berechnen der Anzahl der Samples pro Klasse unter Beibehaltung des Verhältnisses
    car_ratio = len(car_indices) / len(y_data)
    num_car_samples = int(num_samples * car_ratio)
    num_non_car_samples = num_samples - num_car_samples
    
    # Zufällige Auswahl von Indizes für jede Klasse
    selected_car_indices = np.random.choice(car_indices, size=num_car_samples, replace=False)
    selected_non_car_indices = np.random.choice(non_car_indices, size=num_non_car_samples, replace=False)
    
    # Kombinieren der ausgewählten Indizes
    selected_indices = np.concatenate([selected_car_indices, selected_non_car_indices])
    np.random.shuffle(selected_indices)
    
    return x_data[selected_indices], y_data[selected_indices]

# Reduzieren der Trainings- und Testdaten
num_train_samples = 5000  # 10% der ursprünglichen Trainingsdaten
num_test_samples = 1000   # 10% der ursprünglichen Testdaten

x_train_reduced, y_train_reduced = reduce_dataset(x_train, y_train, num_train_samples)
x_test_reduced, y_test_reduced = reduce_dataset(x_test, y_test, num_test_samples)

print(f"Reduzierte Trainingsbilder: {x_train_reduced.shape}")
print(f"Reduzierte Testbilder: {x_test_reduced.shape}")
print(f"Anzahl der Auto-Bilder im reduzierten Trainingsdatensatz: {np.sum(y_train_reduced == 1)}")
print(f"Anzahl der Auto-Bilder im reduzierten Testdatensatz: {np.sum(y_test_reduced == 1)}")

## Implementierung der CNN-Funktionen

Jetzt implementieren wir die grundlegenden Funktionen für unser CNN. Wir beginnen mit den Hilfsfunktionen für die Faltungsoperation, gefolgt von den Implementierungen für die Vorwärts- und Rückwärtspropagierung jeder Schicht.

Die Implementierung folgt dem mathematischen Fundament von CNNs und umfasst alle notwendigen Operationen für das Training und die Vorhersage.

### Hilfsfunktionen für die Faltungsoperation

Die Faltungsoperation ist das Herzstück eines CNN. Bei dieser Operation wird ein Filter (oder Kernel) über das Eingabebild geschoben, und an jeder Position wird das Skalarprodukt zwischen dem Filter und dem entsprechenden Bereich des Bildes berechnet.

Mathematisch kann die Faltungsoperation für ein 2D-Bild $I$ und einen 2D-Filter $K$ wie folgt ausgedrückt werden:

$(I * K)(i, j) = \sum_{m} \sum_{n} I(i+m, j+n) \cdot K(m, n)$

Für Farbbilder mit mehreren Kanälen wird diese Operation für jeden Kanal separat durchgeführt und die Ergebnisse werden summiert.

Wir implementieren zwei Hilfsfunktionen:
1. `im2col`: Transformiert Bildregionen in Spalten einer Matrix, was eine effizientere Berechnung der Faltung ermöglicht
2. `col2im`: Die Umkehroperation von `im2col`, die eine Spaltenmatrix zurück in ein Bild transformiert

In [ ]:
# Hilfsfunktionen für die Faltungsoperation
def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    """
    Transformiert Regionen eines mehrdimensionalen Arrays in Spalten.
    
    Parameter:
    - input_data: Eingabedaten mit Form (N, C, H, W)
    - filter_h: Höhe des Filters
    - filter_w: Breite des Filters
    - stride: Schrittweite der Faltung
    - pad: Anzahl der Padding-Pixel
    
    Rückgabe:
    - col: Transformierte Daten mit Form (N*out_h*out_w, C*filter_h*filter_w)
    """
    N, C, H, W = input_data.shape
    out_h = (H + 2*pad - filter_h)//stride + 1
    out_w = (W + 2*pad - filter_w)//stride + 1

    # Padding hinzufügen
    img = np.pad(input_data, [(0,0), (0,0), (pad, pad), (pad, pad)], 'constant')
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w))

    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)
    return col

def col2im(col, input_shape, filter_h, filter_w, stride=1, pad=0):
    """
    Transformiert Spalten zurück in ein mehrdimensionales Array.
    
    Parameter:
    - col: Transformierte Daten mit Form (N*out_h*out_w, C*filter_h*filter_w)
    - input_shape: Form der Eingabedaten (N, C, H, W)
    - filter_h: Höhe des Filters
    - filter_w: Breite des Filters
    - stride: Schrittweite der Faltung
    - pad: Anzahl der Padding-Pixel
    
    Rückgabe:
    - img: Transformierte Daten mit Form (N, C, H, W)
    """
    N, C, H, W = input_shape
    out_h = (H + 2*pad - filter_h)//stride + 1
    out_w = (W + 2*pad - filter_w)//stride + 1
    col = col.reshape(N, out_h, out_w, C, filter_h, filter_w).transpose(0, 3, 4, 5, 1, 2)

    img = np.zeros((N, C, H + 2*pad + stride - 1, W + 2*pad + stride - 1))
    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            img[:, :, y:y_max:stride, x:x_max:stride] += col[:, :, y, x, :, :]

    return img[:, :, pad:H + pad, pad:W + pad]

### Vorwärtspropagierung für die Faltungsschicht

Die Faltungsschicht ist die charakteristische Komponente eines CNN. In der Vorwärtspropagierung werden Filter über das Eingabebild geschoben, um Merkmale zu extrahieren.

Unsere Implementierung der Faltungsschicht umfasst:
1. Initialisierung der Filter und Bias-Parameter
2. Vorwärtspropagierung mit der Faltungsoperation
3. Rückwärtspropagierung zur Berechnung der Gradienten

Die Filter werden mit einer Xavier-Initialisierung initialisiert, die für tiefe neuronale Netzwerke entwickelt wurde und eine bessere Konvergenz ermöglicht.

In [ ]:
class Convolution:
    def __init__(self, input_channels, filter_num, filter_size, stride=1, pad=0):
        """
        Initialisiert eine Faltungsschicht.
        
        Parameter:
        - input_channels: Anzahl der Eingangskanäle
        - filter_num: Anzahl der Filter
        - filter_size: Größe der Filter (Höhe = Breite)
        - stride: Schrittweite der Faltung
        - pad: Anzahl der Padding-Pixel
        """
        self.input_channels = input_channels
        self.filter_num = filter_num
        self.filter_size = filter_size
        self.filter_h = filter_size
        self.filter_w = filter_size
        self.stride = stride
        self.pad = pad
        
        # Xavier-Initialisierung für bessere Konvergenz
        scale = np.sqrt(2.0 / (input_channels * filter_size * filter_size))
        self.W = scale * np.random.randn(filter_num, input_channels, filter_size, filter_size)
        self.b = np.zeros(filter_num)
        
        # Gradienten
        self.dW = None
        self.db = None
        
        # Für die Rückwärtspropagierung
        self.x = None
        self.col = None
        self.col_W = None
    
    def forward(self, x):
        """
        Vorwärtspropagierung für die Faltungsschicht.
        
        Parameter:
        - x: Eingabedaten mit Form (N, C, H, W)
        
        Rückgabe:
        - out: Ausgabedaten mit Form (N, filter_num, out_h, out_w)
        """
        N, C, H, W = x.shape
        out_h = (H + 2*self.pad - self.filter_h)//self.stride + 1
        out_w = (W + 2*self.pad - self.filter_w)//self.stride + 1
        
        # Transformation der Eingabe für effiziente Faltung
        col = im2col(x, self.filter_h, self.filter_w, self.stride, self.pad)
        col_W = self.W.reshape(self.filter_num, -1).T
        
        # Faltungsoperation als Matrixmultiplikation
        out = np.dot(col, col_W) + self.b
        out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)
        
        # Speichern für die Rückwärtspropagierung
        self.x = x
        self.col = col
        self.col_W = col_W
        
        return out
    
    def backward(self, dout):
        """
        Rückwärtspropagierung für die Faltungsschicht.
        
        Parameter:
        - dout: Gradient der Verlustfunktion bezüglich der Ausgabe
        
        Rückgabe:
        - dx: Gradient der Verlustfunktion bezüglich der Eingabe
        """
        FN, C, FH, FW = self.W.shape
        dout = dout.transpose(0, 2, 3, 1).reshape(-1, FN)
        
        # Berechnung der Gradienten
        self.db = np.sum(dout, axis=0)
        self.dW = np.dot(self.col.T, dout)
        self.dW = self.dW.transpose(1, 0).reshape(FN, C, FH, FW)
        
        # Berechnung des Gradienten bezüglich der Eingabe
        dcol = np.dot(dout, self.col_W.T)
        dx = col2im(dcol, self.x.shape, FH, FW, self.stride, self.pad)
        
        return dx

### Aktivierungsfunktionen

Aktivierungsfunktionen führen Nichtlinearität in das neuronale Netzwerk ein, was es ermöglicht, komplexe Muster zu lernen. Wir implementieren die ReLU-Aktivierungsfunktion (Rectified Linear Unit) und die Sigmoid-Aktivierungsfunktion.

1. **ReLU**: $f(x) = \max(0, x)$
   - Einfach zu berechnen und differenzieren
   - Hilft, das Problem des verschwindenden Gradienten zu mildern
   - Wird in den versteckten Schichten verwendet

2. **Sigmoid**: $f(x) = \frac{1}{1 + e^{-x}}$
   - Bildet Eingaben auf den Bereich (0, 1) ab
   - Geeignet für binäre Klassifikation
   - Wird in der Ausgabeschicht für binäre Klassifikation verwendet

In [ ]:
class ReLU:
    def __init__(self):
        """
        Initialisiert eine ReLU-Aktivierungsschicht.
        """
        self.mask = None
    
    def forward(self, x):
        """
        Vorwärtspropagierung für die ReLU-Aktivierung.
        
        Parameter:
        - x: Eingabedaten
        
        Rückgabe:
        - out: Aktivierte Ausgabedaten
        """
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0
        return out
    
    def backward(self, dout):
        """
        Rückwärtspropagierung für die ReLU-Aktivierung.
        
        Parameter:
        - dout: Gradient der Verlustfunktion bezüglich der Ausgabe
        
        Rückgabe:
        - dx: Gradient der Verlustfunktion bezüglich der Eingabe
        """
        dout[self.mask] = 0
        dx = dout
        return dx

class Sigmoid:
    def __init__(self):
        """
        Initialisiert eine Sigmoid-Aktivierungsschicht.
        """
        self.out = None
    
    def forward(self, x):
        """
        Vorwärtspropagierung für die Sigmoid-Aktivierung.
        
        Parameter:
        - x: Eingabedaten
        
        Rückgabe:
        - out: Aktivierte Ausgabedaten
        """
        out = 1 / (1 + np.exp(-x))
        self.out = out
        return out
    
    def backward(self, dout):
        """
        Rückwärtspropagierung für die Sigmoid-Aktivierung.
        
        Parameter:
        - dout: Gradient der Verlustfunktion bezüglich der Ausgabe
        
        Rückgabe:
        - dx: Gradient der Verlustfunktion bezüglich der Eingabe
        """
        dx = dout * (1.0 - self.out) * self.out
        return dx

### Pooling-Schicht

Die Pooling-Schicht reduziert die räumliche Dimension der Feature Maps, was die Berechnungseffizienz erhöht und eine gewisse Translationsinvarianz einführt. Wir implementieren Max-Pooling, bei dem der maximale Wert aus einem definierten Fenster ausgewählt wird.

Mathematisch kann Max-Pooling für ein Fenster der Größe $h \times w$ wie folgt ausgedrückt werden:

$\text{MaxPool}(i, j) = \max_{0 \leq m < h, 0 \leq n < w} I(i \cdot s + m, j \cdot s + n)$

wobei $s$ die Schrittweite (stride) ist.

In der Rückwärtspropagierung werden die Gradienten nur an die Position des maximalen Wertes im Eingabefenster weitergegeben.

In [ ]:
class Pooling:
    def __init__(self, pool_h, pool_w, stride=1, pad=0):
        """
        Initialisiert eine Pooling-Schicht.
        
        Parameter:
        - pool_h: Höhe des Pooling-Fensters
        - pool_w: Breite des Pooling-Fensters
        - stride: Schrittweite des Pooling
        - pad: Anzahl der Padding-Pixel
        """
        self.pool_h = pool_h
        self.pool_w = pool_w
        self.stride = stride
        self.pad = pad
        
        # Für die Rückwärtspropagierung
        self.x = None
        self.arg_max = None
    
    def forward(self, x):
        """
        Vorwärtspropagierung für die Pooling-Schicht.
        
        Parameter:
        - x: Eingabedaten mit Form (N, C, H, W)
        
        Rückgabe:
        - out: Ausgabedaten mit reduzierter räumlicher Dimension
        """
        N, C, H, W = x.shape
        out_h = (H - self.pool_h) // self.stride + 1
        out_w = (W - self.pool_w) // self.stride + 1
        
        # Transformation der Eingabe für effizientes Pooling
        col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
        col = col.reshape(-1, self.pool_h * self.pool_w)
        
        # Max-Pooling
        arg_max = np.argmax(col, axis=1)
        out = np.max(col, axis=1)
        out = out.reshape(N, out_h, out_w, C).transpose(0, 3, 1, 2)
        
        # Speichern für die Rückwärtspropagierung
        self.x = x
        self.arg_max = arg_max
        
        return out
    
    def backward(self, dout):
        """
        Rückwärtspropagierung für die Pooling-Schicht.
        
        Parameter:
        - dout: Gradient der Verlustfunktion bezüglich der Ausgabe
        
        Rückgabe:
        - dx: Gradient der Verlustfunktion bezüglich der Eingabe
        """
        dout = dout.transpose(0, 2, 3, 1)
        pool_size = self.pool_h * self.pool_w
        dmax = np.zeros((dout.size, pool_size))
        dmax[np.arange(self.arg_max.size), self.arg_max.flatten()] = dout.flatten()
        dmax = dmax.reshape(dout.shape[0], dout.shape[1], dout.shape[2], dout.shape[3], pool_size)
        dcol = dmax.reshape(dmax.shape[0] * dmax.shape[1] * dmax.shape[2] * dmax.shape[3], -1)
        dx = col2im(dcol, self.x.shape, self.pool_h, self.pool_w, self.stride, self.pad)
        
        return dx

### Flatten und Fully Connected Layer

Nach den Faltungs- und Pooling-Schichten müssen wir die mehrdimensionalen Feature Maps in einen eindimensionalen Vektor umwandeln, um sie an vollständig verbundene Schichten zu übergeben. Dies wird durch die Flatten-Schicht erreicht.

Die vollständig verbundene Schicht (Fully Connected Layer) verbindet jedes Neuron mit allen Neuronen der vorherigen Schicht, ähnlich wie in traditionellen neuronalen Netzwerken. Mathematisch kann dies als Matrixmultiplikation ausgedrückt werden:

$y = Wx + b$

wobei $W$ die Gewichtsmatrix, $x$ der Eingabevektor und $b$ der Bias-Vektor ist.

In [ ]:
class Flatten:
    def __init__(self):
        """
        Initialisiert eine Flatten-Schicht.
        """
        self.original_shape = None
    
    def forward(self, x):
        """
        Vorwärtspropagierung für die Flatten-Schicht.
        
        Parameter:
        - x: Eingabedaten mit Form (N, C, H, W)
        
        Rückgabe:
        - out: Ausgabedaten mit Form (N, C*H*W)
        """
        self.original_shape = x.shape
        N = x.shape[0]
        out = x.reshape(N, -1)
        return out
    
    def backward(self, dout):
        """
        Rückwärtspropagierung für die Flatten-Schicht.
        
        Parameter:
        - dout: Gradient der Verlustfunktion bezüglich der Ausgabe
        
        Rückgabe:
        - dx: Gradient der Verlustfunktion bezüglich der Eingabe
        """
        dx = dout.reshape(self.original_shape)
        return dx

class FullyConnected:
    def __init__(self, input_size, output_size):
        """
        Initialisiert eine vollständig verbundene Schicht.
        
        Parameter:
        - input_size: Größe der Eingabe
        - output_size: Größe der Ausgabe
        """
        # Xavier-Initialisierung für bessere Konvergenz
        scale = np.sqrt(2.0 / input_size)
        self.W = scale * np.random.randn(input_size, output_size)
        self.b = np.zeros(output_size)
        
        # Gradienten
        self.dW = None
        self.db = None
        
        # Für die Rückwärtspropagierung
        self.x = None
    
    def forward(self, x):
        """
        Vorwärtspropagierung für die vollständig verbundene Schicht.
        
        Parameter:
        - x: Eingabedaten mit Form (N, input_size)
        
        Rückgabe:
        - out: Ausgabedaten mit Form (N, output_size)
        """
        self.x = x
        out = np.dot(x, self.W) + self.b
        return out
    
    def backward(self, dout):
        """
        Rückwärtspropagierung für die vollständig verbundene Schicht.
        
        Parameter:
        - dout: Gradient der Verlustfunktion bezüglich der Ausgabe
        
        Rückgabe:
        - dx: Gradient der Verlustfunktion bezüglich der Eingabe
        """
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        return dx

### Kostenfunktion

Für unser binäres Klassifikationsproblem verwenden wir die binäre Kreuzentropie-Verlustfunktion (Binary Cross-Entropy Loss). Diese Funktion misst den Unterschied zwischen den vorhergesagten Wahrscheinlichkeiten und den tatsächlichen Labels.

Mathematisch ist die binäre Kreuzentropie-Verlustfunktion für ein einzelnes Beispiel definiert als:

$L(y, \hat{y}) = -[y \log(\hat{y}) + (1-y) \log(1-\hat{y})]$

wobei $y$ das tatsächliche Label (0 oder 1) und $\hat{y}$ die vorhergesagte Wahrscheinlichkeit ist.

Für einen Batch von Beispielen berechnen wir den Durchschnitt der Verluste.

In [ ]:
class BinaryCrossEntropyLoss:
    def __init__(self):
        """
        Initialisiert eine binäre Kreuzentropie-Verlustfunktion.
        """
        self.loss = None
        self.y = None
        self.t = None
    
    def forward(self, y, t):
        """
        Berechnet den Verlust.
        
        Parameter:
        - y: Vorhergesagte Wahrscheinlichkeiten mit Form (N, 1)
        - t: Tatsächliche Labels mit Form (N, 1)
        
        Rückgabe:
        - loss: Durchschnittlicher Verlust
        """
        self.y = y
        self.t = t
        
        # Numerische Stabilität
        epsilon = 1e-7
        y = np.clip(y, epsilon, 1 - epsilon)
        
        # Binäre Kreuzentropie
        self.loss = -np.sum(t * np.log(y) + (1 - t) * np.log(1 - y)) / len(y)
        
        return self.loss
    
    def backward(self, dout=1):
        """
        Berechnet den Gradienten der Verlustfunktion bezüglich der Eingabe.
        
        Parameter:
        - dout: Upstream-Gradient (normalerweise 1)
        
        Rückgabe:
        - dx: Gradient der Verlustfunktion bezüglich der Eingabe
        """
        batch_size = self.t.shape[0]
        
        # Gradient der binären Kreuzentropie
        dx = (self.y - self.t) / batch_size
        
        return dx

### Rückwärtspropagierung

Die Rückwärtspropagierung ist der Algorithmus, mit dem die Gradienten der Verlustfunktion in Bezug auf die Parameter des Netzwerks berechnet werden. Diese Gradienten werden dann verwendet, um die Parameter durch Gradientenabstieg zu aktualisieren.

Der Algorithmus arbeitet rückwärts durch das Netzwerk, beginnend mit der Ausgabeschicht, und berechnet die Gradienten für jede Schicht basierend auf der Kettenregel der Differentiation.

Für jede Schicht haben wir bereits die Rückwärtspropagierungsmethode implementiert, die den Gradienten der Verlustfunktion bezüglich der Eingabe der Schicht berechnet und die Gradienten bezüglich der Parameter der Schicht aktualisiert.

### Vorwärts- und Rückwärtspropagierung für das gesamte Modell

Jetzt kombinieren wir alle implementierten Schichten zu einem vollständigen CNN-Modell. Das Modell führt die Vorwärtspropagierung durch, um Vorhersagen zu treffen, und die Rückwärtspropagierung, um die Gradienten zu berechnen.

Unsere CNN-Architektur besteht aus:
1. Einer Faltungsschicht mit ReLU-Aktivierung
2. Einer Max-Pooling-Schicht
3. Einer zweiten Faltungsschicht mit ReLU-Aktivierung
4. Einer zweiten Max-Pooling-Schicht
5. Einer Flatten-Schicht
6. Einer vollständig verbundenen Schicht mit ReLU-Aktivierung
7. Einer Ausgabeschicht mit Sigmoid-Aktivierung für binäre Klassifikation

In [ ]:
class CNN:
    def __init__(self, input_shape, conv_param_1, conv_param_2, hidden_size, output_size):
        """
        Initialisiert ein CNN-Modell.
        
        Parameter:
        - input_shape: Form der Eingabedaten (C, H, W)
        - conv_param_1: Parameter für die erste Faltungsschicht
        - conv_param_2: Parameter für die zweite Faltungsschicht
        - hidden_size: Größe der versteckten vollständig verbundenen Schicht
        - output_size: Größe der Ausgabe (1 für binäre Klassifikation)
        """
        # Extrahieren der Parameter
        C, H, W = input_shape
        filter_num_1, filter_size_1, stride_1, pad_1 = conv_param_1
        filter_num_2, filter_size_2, stride_2, pad_2 = conv_param_2
        
        # Berechnung der Ausgabegrößen der Faltungs- und Pooling-Schichten
        conv_output_size_1 = (H - filter_size_1 + 2*pad_1) // stride_1 + 1
        pool_output_size_1 = (conv_output_size_1 - 2) // 2 + 1
        conv_output_size_2 = (pool_output_size_1 - filter_size_2 + 2*pad_2) // stride_2 + 1
        pool_output_size_2 = (conv_output_size_2 - 2) // 2 + 1
        
        # Berechnung der Eingabegröße für die vollständig verbundene Schicht
        fc_input_size = filter_num_2 * pool_output_size_2 * pool_output_size_2
        
        # Initialisierung der Schichten
        self.layers = [
            Convolution(C, filter_num_1, filter_size_1, stride_1, pad_1),
            ReLU(),
            Pooling(2, 2, 2),
            Convolution(filter_num_1, filter_num_2, filter_size_2, stride_2, pad_2),
            ReLU(),
            Pooling(2, 2, 2),
            Flatten(),
            FullyConnected(fc_input_size, hidden_size),
            ReLU(),
            FullyConnected(hidden_size, output_size),
            Sigmoid()
        ]
        
        # Verlustfunktion
        self.loss_layer = BinaryCrossEntropyLoss()
        
        # Parameter und Gradienten
        self.params, self.grads = [], []
        for layer in self.layers:
            if hasattr(layer, 'W'):
                self.params.append(layer.W)
                self.grads.append(layer.dW)
            if hasattr(layer, 'b'):
                self.params.append(layer.b)
                self.grads.append(layer.db)
    
    def predict(self, x):
        """
        Macht Vorhersagen für die Eingabedaten.
        
        Parameter:
        - x: Eingabedaten mit Form (N, C, H, W)
        
        Rückgabe:
        - y: Vorhergesagte Wahrscheinlichkeiten mit Form (N, 1)
        """
        for layer in self.layers:
            x = layer.forward(x)
        return x
    
    def forward(self, x, t):
        """
        Vorwärtspropagierung und Berechnung des Verlusts.
        
        Parameter:
        - x: Eingabedaten mit Form (N, C, H, W)
        - t: Tatsächliche Labels mit Form (N, 1)
        
        Rückgabe:
        - loss: Verlust
        """
        y = self.predict(x)
        loss = self.loss_layer.forward(y, t)
        return loss
    
    def backward(self):
        """
        Rückwärtspropagierung zur Berechnung der Gradienten.
        
        Rückgabe:
        - dout: Gradient der Verlustfunktion bezüglich der Eingabe
        """
        dout = self.loss_layer.backward()
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout
    
    def accuracy(self, x, t):
        """
        Berechnet die Genauigkeit der Vorhersagen.
        
        Parameter:
        - x: Eingabedaten mit Form (N, C, H, W)
        - t: Tatsächliche Labels mit Form (N, 1)
        
        Rückgabe:
        - accuracy: Genauigkeit (0-1)
        """
        y = self.predict(x)
        y = (y > 0.5).astype(int)
        accuracy = np.mean(y == t)
        return accuracy

### Aktualisierung der Parameter und Vorhersage

Nach der Berechnung der Gradienten müssen wir die Parameter des Netzwerks aktualisieren. Wir implementieren den Gradientenabstieg-Optimierer, der die Parameter in Richtung des negativen Gradienten aktualisiert.

Die Aktualisierungsregel für einen Parameter $\theta$ ist:

$\theta = \theta - \eta \cdot \nabla_\theta L$

wobei $\eta$ die Lernrate und $\nabla_\theta L$ der Gradient der Verlustfunktion bezüglich des Parameters ist.

In [ ]:
class SGD:
    def __init__(self, lr=0.01):
        """
        Initialisiert einen Stochastic Gradient Descent Optimierer.
        
        Parameter:
        - lr: Lernrate
        """
        self.lr = lr
    
    def update(self, params, grads):
        """
        Aktualisiert die Parameter basierend auf den Gradienten.
        
        Parameter:
        - params: Liste der Parameter
        - grads: Liste der Gradienten
        """
        for i in range(len(params)):
            params[i] -= self.lr * grads[i]

### Mini-Batch Gradient Descent

Statt den Gradientenabstieg auf dem gesamten Datensatz durchzuführen, verwenden wir Mini-Batch Gradient Descent, bei dem wir in jeder Iteration nur eine kleine Teilmenge (Batch) der Daten verwenden. Dies beschleunigt das Training und kann zu einer besseren Konvergenz führen.

Wir implementieren eine Funktion, die die Daten in Batches aufteilt und für jeden Batch die Vorwärts- und Rückwärtspropagierung durchführt, gefolgt von einer Aktualisierung der Parameter.

In [ ]:
def train_step(model, optimizer, x_batch, t_batch):
    """
    Führt einen Trainingsschritt für einen Batch durch.
    
    Parameter:
    - model: CNN-Modell
    - optimizer: Optimierer
    - x_batch: Batch von Eingabedaten
    - t_batch: Batch von Labels
    
    Rückgabe:
    - loss: Verlust für den Batch
    """
    # Vorwärtspropagierung
    loss = model.forward(x_batch, t_batch)
    
    # Rückwärtspropagierung
    model.backward()
    
    # Aktualisierung der Parameter
    optimizer.update(model.params, model.grads)
    
    return loss

def generate_batches(x, t, batch_size):
    """
    Generiert Batches aus den Daten.
    
    Parameter:
    - x: Eingabedaten
    - t: Labels
    - batch_size: Größe der Batches
    
    Rückgabe:
    - batches: Liste von (x_batch, t_batch) Tupeln
    """
    N = x.shape[0]
    indices = np.arange(N)
    np.random.shuffle(indices)
    
    batches = []
    for i in range(0, N, batch_size):
        batch_indices = indices[i:i+batch_size]
        x_batch = x[batch_indices]
        t_batch = t[batch_indices]
        batches.append((x_batch, t_batch))
    
    return batches

### Hauptfunktion für das Training des Modells

Schließlich implementieren wir die Hauptfunktion für das Training des Modells. Diese Funktion initialisiert das Modell, führt das Training für eine bestimmte Anzahl von Epochen durch und evaluiert das Modell regelmäßig auf den Validierungsdaten.

Wir speichern auch den Trainingsverlauf, um später die Leistung des Modells zu visualisieren.

In [ ]:
def train_model(x_train, t_train, x_val, t_val, input_shape, conv_param_1, conv_param_2, hidden_size, output_size, 
                batch_size=32, epochs=10, lr=0.01, verbose=True):
    """
    Trainiert ein CNN-Modell.
    
    Parameter:
    - x_train: Trainingsdaten
    - t_train: Trainings-Labels
    - x_val: Validierungsdaten
    - t_val: Validierungs-Labels
    - input_shape: Form der Eingabedaten (C, H, W)
    - conv_param_1: Parameter für die erste Faltungsschicht
    - conv_param_2: Parameter für die zweite Faltungsschicht
    - hidden_size: Größe der versteckten vollständig verbundenen Schicht
    - output_size: Größe der Ausgabe (1 für binäre Klassifikation)
    - batch_size: Größe der Batches
    - epochs: Anzahl der Trainingsepochen
    - lr: Lernrate
    - verbose: Ob Fortschrittsinformationen angezeigt werden sollen
    
    Rückgabe:
    - model: Trainiertes Modell
    - history: Trainingsverlauf
    """
    # Initialisierung des Modells und des Optimierers
    model = CNN(input_shape, conv_param_1, conv_param_2, hidden_size, output_size)
    optimizer = SGD(lr=lr)
    
    # Trainingsverlauf
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    # Training
    for epoch in range(epochs):
        start_time = time.time()
        
        # Generieren der Batches
        batches = generate_batches(x_train, t_train, batch_size)
        
        # Training für jeden Batch
        train_loss = 0
        for x_batch, t_batch in batches:
            loss = train_step(model, optimizer, x_batch, t_batch)
            train_loss += loss
        
        # Berechnung des durchschnittlichen Verlusts und der Genauigkeit
        train_loss /= len(batches)
        train_acc = model.accuracy(x_train, t_train)
        
        # Evaluierung auf den Validierungsdaten
        val_loss = model.forward(x_val, t_val)
        val_acc = model.accuracy(x_val, t_val)
        
        # Speichern des Trainingsverlaufs
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Ausgabe des Fortschritts
        if verbose:
            elapsed_time = time.time() - start_time
            print(f"Epoch {epoch+1}/{epochs} - {elapsed_time:.2f}s - train_loss: {train_loss:.4f} - train_acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")
    
    return model, history

## Training des Modells

Jetzt trainieren wir unser benutzerdefiniertes CNN-Modell auf den reduzierten Daten. Wir definieren die Hyperparameter und die Architektur des Modells und starten das Training.

Da unsere Implementierung nicht optimiert ist, kann das Training einige Zeit in Anspruch nehmen, selbst auf dem reduzierten Datensatz.

In [ ]:
# Vorbereitung der Daten
# Umformen der Daten von (N, H, W, C) zu (N, C, H, W) für unsere Implementierung
x_train_reshaped = x_train_reduced.transpose(0, 3, 1, 2)
x_test_reshaped = x_test_reduced.transpose(0, 3, 1, 2)

# Hyperparameter
input_shape = (3, 32, 32)  # (C, H, W)
conv_param_1 = (16, 3, 1, 1)  # (filter_num, filter_size, stride, pad)
conv_param_2 = (32, 3, 1, 1)  # (filter_num, filter_size, stride, pad)
hidden_size = 64
output_size = 1
batch_size = 32
epochs = 5
lr = 0.01

# Aufteilung in Trainings- und Validierungsdaten
validation_split = 0.2
n_val = int(len(x_train_reshaped) * validation_split)
x_val = x_train_reshaped[:n_val]
t_val = y_train_reduced[:n_val]
x_train_final = x_train_reshaped[n_val:]
t_train_final = y_train_reduced[n_val:]

print(f"Trainingsbilder: {x_train_final.shape}")
print(f"Validierungsbilder: {x_val.shape}")
print(f"Testbilder: {x_test_reshaped.shape}")

# Training des Modells
model, history = train_model(
    x_train_final, t_train_final, x_val, t_val,
    input_shape, conv_param_1, conv_param_2, hidden_size, output_size,
    batch_size=batch_size, epochs=epochs, lr=lr, verbose=True
)

## Evaluierung des Modells

Nach dem Training evaluieren wir das Modell auf den Testdaten, um seine Generalisierungsfähigkeit zu bewerten. Wir berechnen den Verlust und die Genauigkeit und visualisieren den Trainingsverlauf.

In [ ]:
# Evaluierung auf den Testdaten
test_loss = model.forward(x_test_reshaped, y_test_reduced)
test_acc = model.accuracy(x_test_reshaped, y_test_reduced)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Visualisierung des Trainingsverlaufs
plt.figure(figsize=(12, 5))

# Plot für den Verlust
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Trainingsverlauf - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot für die Genauigkeit
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Training Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.title('Trainingsverlauf - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(visualizations_dir, 'custom_cnn_training_history.png'))
plt.show()

## Visualisierung der Vorhersagen

Um ein besseres Verständnis für die Leistung unseres Modells zu bekommen, visualisieren wir einige Beispiele aus dem Testdatensatz zusammen mit den Vorhersagen des Modells. Wir zeigen sowohl korrekte als auch falsche Vorhersagen.

In [ ]:
# Vorhersagen für die Testdaten
y_pred_prob = model.predict(x_test_reshaped)
y_pred = (y_pred_prob > 0.5).astype(int)

# Visualisierung einiger Vorhersagen
def visualize_predictions(x_data, y_true, y_pred, y_pred_prob, num_examples=10):
    # Umformen der Daten zurück zu (N, H, W, C) für die Visualisierung
    x_data_vis = x_data.transpose(0, 2, 3, 1)
    
    # Zufällige Indizes auswählen
    np.random.seed(42)  # Für Reproduzierbarkeit
    indices = np.random.choice(len(y_true), size=num_examples, replace=False)
    
    # Erstellen der Visualisierung
    plt.figure(figsize=(15, 8))
    for i, idx in enumerate(indices):
        plt.subplot(2, 5, i+1)
        plt.imshow(x_data_vis[idx])
        plt.axis('off')
        
        true_label = 'Auto' if y_true[idx][0] == 1 else 'Nicht-Auto'
        pred_label = 'Auto' if y_pred[idx][0] == 1 else 'Nicht-Auto'
        confidence = y_pred_prob[idx][0]
        
        color = 'green' if y_true[idx][0] == y_pred[idx][0] else 'red'
        plt.title(f"Wahr: {true_label}\nVorhersage: {pred_label}\nKonfidenz: {confidence:.2f}", color=color)
    
    plt.tight_layout()
    plt.savefig(os.path.join(visualizations_dir, 'custom_cnn_predictions.png'))
    plt.show()

# Visualisierung von Vorhersagen
visualize_predictions(x_test_reshaped, y_test_reduced, y_pred, y_pred_prob)

## Zusammenfassung

In diesem Notebook haben wir ein Convolutional Neural Network (CNN) von Grund auf implementiert, ohne die höheren Abstraktionen von keras.models oder keras.layers zu verwenden. Hier sind die wichtigsten Punkte:

1. **Implementierung der grundlegenden Operationen**: Wir haben alle grundlegenden Operationen eines CNN implementiert, einschließlich Faltung, Aktivierungsfunktionen, Pooling und vollständig verbundene Schichten.

2. **Vorwärts- und Rückwärtspropagierung**: Wir haben die Vorwärtspropagierung für Vorhersagen und die Rückwärtspropagierung für die Berechnung der Gradienten implementiert.

3. **Training mit Mini-Batch Gradient Descent**: Wir haben das Modell mit Mini-Batch Gradient Descent trainiert, um die Effizienz zu verbessern.

4. **Evaluierung und Visualisierung**: Wir haben das Modell auf Testdaten evaluiert und die Ergebnisse visualisiert.

Diese Implementierung bietet ein tieferes Verständnis für die innere Funktionsweise von CNNs, ist aber weniger effizient als optimierte Bibliotheken wie TensorFlow. In der Praxis würde man für Produktionsanwendungen optimierte Bibliotheken verwenden, aber die Implementierung von Grund auf ist ein wertvolles Lernwerkzeug.

Im nächsten Notebook werden wir ein vortrainiertes CNN laden und für unsere Autoerkennungsaufgabe anpassen.